# ZARX-1B Training Notebook (Google Colab)
1B parameter code + ARC reasoning model trained from scratch.

**GPU:** T4 (16GB VRAM) | **Runtime:** 4-12hrs per session

## Persistence (NEVER blocks the process!)
- Google Drive: saves data + checkpoints (if space available, fails gracefully)
- GitHub: pushes small files to codedbytahir/zarx-checkpoints (<95MB)
- HuggingFace: pushes everything to Chvigo/zarx-checks (no size limit)
- If Drive is full, automatically falls back to GitHub + HuggingFace

## Setup
1. Go to Runtime -> Change runtime type -> T4 GPU
2. Run cells in order
3. Training auto-resumes from checkpoints

In [ ]:
#@title 0. Mount Google Drive (MUST DO FIRST!)
from google.colab import drive
drive.mount('/content/drive')

# Create persistence directories
import os, shutil
DRIVE_DIR = '/content/drive/MyDrive/ZARX-1B'
try:
    for d in ['data/raw/prox-code', 'data/raw/arc', 'data/processed',
              'checkpoints', 'tokenizer', 'configs']:
        os.makedirs(f'{DRIVE_DIR}/{d}', exist_ok=True)
    # Check Drive space
    stat = shutil.disk_usage(DRIVE_DIR)
    free_gb = stat.free / 1e9
    print(f'Drive mounted! Free space: {free_gb:.1f}GB')
    if free_gb < 1:
        print(f'WARNING: Drive is nearly full! Will use GitHub+HF fallback.')
except OSError as e:
    print(f'Drive mount/mkdir failed: {e}')
    print(f'Will use GitHub+HuggingFace for persistence instead.')

In [ ]:
#@title 1. Install Dependencies
!pip install -q torch>=2.1.0 bitsandbytes wandb huggingface_hub
!pip install -q datasets tokenizers accelerate datasketch
# Flash Attention skipped on T4 (not supported, SDPA auto-fallback works)
import torch
if torch.cuda.is_available() and ('A100' in torch.cuda.get_device_name() or 'L4' in torch.cuda.get_device_name()):
    !pip install -q flash-attn --no-build-isolation
print('Dependencies installed!')

In [ ]:
#@title 2. Set Credentials
import os

# HuggingFace login
HF_TOKEN = '' #@param {type:"string"}
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)

# W&B login
WANDB_KEY = '' #@param {type:"string"}
if WANDB_KEY:
    import wandb
    wandb.login(key=WANDB_KEY)

# GitHub token for checkpoint pushes
GH_TOKEN = '' #@param {type:"string"}

print('Logins complete!')

In [ ]:
#@title 3. Clone ZARX-1B Repository
%cd /content
!rm -rf ZARX-1B
!git clone https://github.com/codedbytahir/ZARX-1B.git
%cd ZARX-1B
# Also clone checkpoint repo
if GH_TOKEN:
    !git clone https://{GH_TOKEN}@github.com/codedbytahir/zarx-checkpoints.git /content/zarx-checkpoints
print('Repository cloned!')

In [ ]:
#@title 4. Verify GPU & Test Model
import torch
import sys
sys.path.insert(0, '.')

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'BF16 supported: {torch.cuda.is_bf16_supported()}')

from src.model import build_model
model = build_model('configs/model_config.json')
model = model.cuda()
x = torch.randint(0, 32000, (1, 128)).cuda()
out = model(x)
print(f'Forward pass OK! Output shape: {out["logits"].shape}')
print(f'GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB used')
del model, x, out
torch.cuda.empty_cache()

In [ ]:
#@title 5A. Option A: Run Full Day 1 Setup (first time only)
# This downloads data, processes it, trains tokenizer, and runs test training.
# Every step tries Drive first, then GitHub+HF fallback. NEVER stops due to storage.

!python scripts/setup_colab_day1.py \
    --hf_token "{HF_TOKEN}" \
    --wandb_key "{WANDB_KEY}" \
    --hf_repo "Chvigo/zarx-checks" \
    --github_token "{GH_TOKEN}" \
    --github_repo "codedbytahir/zarx-checkpoints"

In [ ]:
#@title 5B. Option B: Restore from Drive/HF + Start Training (daily routine)
# If you already ran Day 1 setup, this restores data and starts training.
# Tries Drive first, then HuggingFace if Drive is full.

import shutil
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/ZARX-1B')
DATA = Path('/content/data')
DATA.mkdir(parents=True, exist_ok=True)

# Restore processed data
restored = False
if (DRIVE / 'data' / 'processed').exists() and any((DRIVE / 'data' / 'processed').glob('*.jsonl')):
    try:
        print('Restoring processed data from Drive...')
        shutil.copytree(str(DRIVE / 'data' / 'processed'), str(DATA / 'processed'), dirs_exist_ok=True)
        print('Data restored from Drive!')
        restored = True
    except Exception as e:
        print(f'Drive restore failed: {e}')

if not restored and HF_TOKEN:
    print('Trying to restore from HuggingFace...')
    from huggingface_hub import snapshot_download
    try:
        snapshot_download(
            repo_id='Chvigo/zarx-checks',
            allow_patterns=['data/processed/*'],
            repo_type='model',
            token=HF_TOKEN,
            local_dir=str(DATA),
        )
        print('Data restored from HuggingFace!')
        restored = True
    except Exception as e:
        print(f'HF restore failed: {e}')

if not restored:
    print('No processed data found! Run Day 1 setup first (Cell 5A).')

# Restore tokenizer
tokenizer_restored = False
if (DRIVE / 'tokenizer' / 'tokenizer.json').exists():
    try:
        shutil.copy2(str(DRIVE / 'tokenizer' / 'tokenizer.json'), 'configs/tokenizer.json')
        print('Tokenizer restored from Drive!')
        tokenizer_restored = True
    except Exception:
        pass

if not tokenizer_restored and HF_TOKEN:
    from huggingface_hub import hf_hub_download
    try:
        hf_hub_download(
            repo_id='Chvigo/zarx-checks',
            filename='data/tokenizer/tokenizer.json',
            repo_type='model',
            token=HF_TOKEN,
            local_dir='.',
        )
        print('Tokenizer restored from HuggingFace!')
    except Exception:
        pass

# Restore checkpoints (for auto-resume)
ckpt_restored = False
if (DRIVE / 'checkpoints').exists() and any((DRIVE / 'checkpoints').glob('*.pt')):
    try:
        print('Restoring checkpoints from Drive...')
        shutil.copytree(str(DRIVE / 'checkpoints'), '/content/checkpoints', dirs_exist_ok=True)
        print('Checkpoints restored! Training will auto-resume.')
        ckpt_restored = True
    except Exception as e:
        print(f'Drive checkpoint restore failed: {e}')

if not ckpt_restored:
    os.makedirs('/content/checkpoints', exist_ok=True)

# Start training
!python src/train.py \
    --model_config configs/model_config.json \
    --tokenizer_path configs/tokenizer.json \
    --data_path /content/data/processed \
    --micro_batch_size 1 \
    --gradient_accumulation_steps 32 \
    --learning_rate 3e-4 \
    --warmup_steps 2000 \
    --total_tokens 10000000000 \
    --use_8bit_adam \
    --checkpoint_dir /content/checkpoints \
    --drive_dir /content/drive/MyDrive/ZARX-1B \
    --github_token "{GH_TOKEN}" \
    --github_repo codedbytahir/zarx-checkpoints \
    --hf_repo_id Chvigo/zarx-checks \
    --hf_token "{HF_TOKEN}" \
    --save_every_local 100 \
    --save_every_drive 500 \
    --save_every_github 1000 \
    --save_every_hf 1000 \
    --log_every_steps 10 \
    --wandb_project zarx-1b \
    --output_dir /content/zarx-1b-final

In [ ]:
#@title 6. EMERGENCY SAVE - Run this if session is about to die!
# One-click save everything to Drive + GitHub + HuggingFace
# NEVER fails the process - if Drive is full, pushes to GitHub+HF instead

import shutil, os
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/ZARX-1B')

# Try saving to Drive (may fail if full)
drive_ok = True
try:
    if Path('/content/checkpoints').exists():
        shutil.copytree('/content/checkpoints', str(DRIVE / 'checkpoints'), dirs_exist_ok=True)
        print('Checkpoints saved to Drive!')
    if Path('/content/data/processed').exists():
        shutil.copytree('/content/data/processed', str(DRIVE / 'data' / 'processed'), dirs_exist_ok=True)
        print('Processed data saved to Drive!')
    if Path('configs/tokenizer.json').exists():
        shutil.copy2('configs/tokenizer.json', str(DRIVE / 'tokenizer' / 'tokenizer.json'))
        print('Tokenizer saved to Drive!')
except OSError as e:
    if 'No space left' in str(e) or 'Quota exceeded' in str(e):
        print(f'DRIVE IS FULL! Falling back to GitHub+HuggingFace...')
        drive_ok = False
    else:
        print(f'Drive save error: {e}')

# Push to GitHub (small files only, <95MB)
if GH_TOKEN and Path('/content/zarx-checkpoints').exists():
    try:
        # Copy checkpoint metadata (small files)
        if Path('/content/checkpoints/training_metadata.json').exists():
            shutil.copy2('/content/checkpoints/training_metadata.json', '/content/zarx-checkpoints/')
        !cd /content/zarx-checkpoints && git add -A && git commit -m 'emergency checkpoint save' && git push https://{GH_TOKEN}@github.com/codedbytahir/zarx-checkpoints.git main
        print('Pushed to GitHub!')
    except Exception as e:
        print(f'GitHub push failed: {e}')

# Push to HuggingFace (no size limit - saves large checkpoints!)
if HF_TOKEN:
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        if Path('/content/checkpoints/checkpoint-latest.pt').exists():
            api.upload_file(
                path_or_fileobj='/content/checkpoints/checkpoint-latest.pt',
                path_in_repo='checkpoint-latest.pt',
                repo_id='Chvigo/zarx-checks',
                repo_type='model',
            )
        if Path('/content/checkpoints/training_metadata.json').exists():
            api.upload_file(
                path_or_fileobj='/content/checkpoints/training_metadata.json',
                path_in_repo='training_metadata.json',
                repo_id='Chvigo/zarx-checks',
                repo_type='model',
            )
        print('Pushed to HuggingFace!')
    except Exception as e:
        print(f'HF push failed: {e}')

print('\nEMERGENCY SAVE COMPLETE! Everything is persisted.')